In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# §6.24 Time-Dependent Perturbation Theory and Fermi's Golden Rule

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.24",
    title="Time-Dependent Perturbation Theory and Fermi's Golden Rule",
    blurb="How quantum systems change. A perturbation that switches on lets "
    "probability flow between states, and the rate of that flow — sharply "
    "favouring transitions that conserve energy, steady when the destination is "
    "a continuum — is Fermi's golden rule, the formula behind nearly all of "
    "spectroscopy. We derive it, watch a reversible oscillation become "
    "irreversible decay, and compute from the hydrogen orbitals themselves the "
    "selection rules that decide which spectral lines an atom is allowed to emit.",
    difficulty="advanced",
    estimate="175–215 min",
)

## Notebook overview

Notebook [§6.21](perturbation-fine-structure.ipynb) asked how a *static* perturbation shifts the energy levels. This
one asks the complementary and more dynamical question: how does a perturbation
that *switches on* drive **transitions** between states? Once $H$ depends on time
the energy eigenstates of $H_0$ are no longer stationary — probability flows
between them — and the object of interest is the transition probability
$P_{i\to f}(t)=|c_f(t)|^2$.

First-order time-dependent perturbation theory gives a strikingly clean answer:
the amplitude to hop from $|i\rangle$ to $|f\rangle$ is the **Fourier transform**
of the perturbation's matrix element at the transition frequency $\omega_{fi}=
(E_f-E_i)/\hbar$. A perturbation oscillating at frequency $\omega$ drives the
transition only near **resonance**, $\hbar\omega\approx E_f-E_i$, with a
$\mathrm{sinc}^2$ lineshape that narrows as the perturbation acts longer — energy
conservation emerging dynamically. For a single discrete final state the
probability *oscillates* (the short-time face of the Rabi problem of [§6.7](time-evolution.ipynb)); but
when the final states form a **continuum**, summing the $\mathrm{sinc}^2$ over
their density $\rho(E)$ turns the oscillation into a probability **linear in
time** — a constant rate. That is **Fermi's golden rule**,
$\Gamma_{i\to f}=(2\pi/\hbar)|\langle f|V|i\rangle|^2\rho(E_f)$, one of the most
used formulas in physics. The crossover from reversible oscillation to
irreversible steady decay, as the destination goes from one level to a continuum,
is the same emergence of irreversibility from reversible dynamics we met in
Volume V ([§5.11](../05-classical-stat-mech/taste-of-nonequilibrium.ipynb)).

Applied to an atom in an oscillating electric field, the perturbation is the
electric-**dipole** coupling and the matrix element is $\langle f|\mathbf r|i
\rangle$. Its angular part — an integral of three spherical harmonics — vanishes
unless $\Delta l=\pm1$ and $\Delta m=0,\pm1$: the **selection rules**, which we
compute directly from the hydrogen orbitals ([§6.15](orbital-angular-momentum.ipynb), [§6.17](hydrogen-atom.ipynb)) rather than invoking
group theory. These are why spectral lines appear where they do, and why some are
missing. We keep the treatment **semiclassical** (classical field, quantized
atom); spontaneous emission, the quantized field, and the Einstein $A/B$
coefficients are named as Volume VII horizons, not developed here.

> **Method specificity.** The first-order amplitude is a time integral by
> `scipy.integrate.quad`; the continuum sum over final states is `numpy.trapezoid`
> against a density of states; the dipole angular integrals use
> `scipy.special.sph_harm_y`. Selection rules are classified with a clean
> tolerance ($10^{-3}$): on this uniform grid the trapezoid rule leaves a truly
> forbidden integral at machine precision ($\sim10^{-16}$), far below the allowed
> values, so the threshold separates them with enormous margin.

> **How to read the checks.** A ✗ is a prompt to locate a discrepancy — an error,
> a convention, or too tight a tolerance — not a verdict that the physics is
> wrong. Passing is strong evidence, not proof.

## Theory in brief

### The time-dependent problem

With $H=H_0+V(t)$, expand the state in the (known) eigenstates of $H_0$,
$|\psi(t)\rangle=\sum_n c_n(t)\,e^{-iE_n t/\hbar}|n\rangle$. The perturbation makes
the coefficients evolve, so probability flows between the levels; the transition
probability is

```{math}
:label: eq-td-setup
P_{i\to f}(t) = |c_f(t)|^2 , \qquad H = H_0 + V(t) .
```

### First-order transition amplitude

To first order in $V$, starting in $|i\rangle$ ($c_i\approx1$), integrating the
Schrödinger equation for the coefficients gives

```{math}
:label: eq-td-amplitude
c_f(t) = -\frac{i}{\hbar}\int_0^t \langle f|V(t')|i\rangle\,
e^{i\omega_{fi}t'}\,\mathrm dt' , \qquad \omega_{fi}=\frac{E_f-E_i}{\hbar} .
```

The amplitude is the **Fourier transform** of the matrix element at the
transition frequency: a transition is driven only if $V(t)$ contains the
frequency $\omega_{fi}$.

### Harmonic perturbation and resonance

For $V(t)=V_0\cos(\omega t)$ switched on for a time $T$, {eq}`eq-td-amplitude`
is sharply peaked when $\omega\approx\omega_{fi}$ (absorption) or
$\omega\approx-\omega_{fi}$ (stimulated emission). Near resonance the probability
is

```{math}
:label: eq-td-resonance
P_{i\to f}(T) = \frac{|V_{0,fi}|^2}{4\hbar^2}\,T^2\,
\mathrm{sinc}^2\!\left[\frac{(\omega-\omega_{fi})T}{2}\right] ,
```

with $\mathrm{sinc}(x)=\sin(x)/x$. The lineshape narrows as $\sim2\pi/T$: the
longer the perturbation acts, the more sharply it enforces energy conservation
$\hbar\omega=E_f-E_i$, becoming a delta function $\delta(E_f-E_i\mp\hbar\omega)$ as
$T\to\infty$.

### Discrete versus continuum

For a **single** discrete final state, $P(T)$ near resonance grows as $T^2$ and
then oscillates — the short-time face of the Rabi problem ([§6.7](time-evolution.ipynb)), and first-order
theory holds only while $P$ stays small.

```{math}
:label: eq-td-discrete-continuum
\text{discrete: } P(T)\ \text{oscillates}; \qquad
\text{continuum: } P(T)=\int \rho(E_f)\,|c_f(T)|^2\,\mathrm dE_f \ \propto\ T .
```

When the final states form a **continuum** of density $\rho(E)$, summing the
$\mathrm{sinc}^2$ over final energies (using $\int\mathrm{sinc}^2(xT/2)\,\mathrm
dx=2\pi/T$) turns the $T^2$ peak into a probability **linear in $T$** — a constant
rate. This crossover from reversible oscillation to irreversible steady decay is
the conceptual heart (echoing [§5.11](../05-classical-stat-mech/taste-of-nonequilibrium.ipynb)).

### Fermi's golden rule

The transition **rate** into a continuum is therefore

```{math}
:label: eq-golden-rule
\Gamma_{i\to f} = \frac{2\pi}{\hbar}\,|\langle f|V|i\rangle|^2\,\rho(E_f) ,
```

evaluated at the final states that conserve energy. The coupling squared times
the density of available final states: it governs decay, scattering, absorption
and emission — anywhere a discrete state couples to a continuum.

### Absorption, emission, and selection rules

For an atom in a field $\mathbf E\cos(\omega t)$ the perturbation is the
electric-dipole coupling

```{math}
:label: eq-radiation
V(t) = -e\,\mathbf E\cdot\mathbf r\,\cos(\omega t) ,
```

so the matrix element is the **dipole moment** $\langle f|\mathbf r|i\rangle$.
Because the components of $\mathbf r$ are proportional to the $Y_1^\mu$, the
angular part of the matrix element is an integral of three spherical harmonics,
and it

```{math}
:label: eq-selection-rules
\int Y_{l'm'}^{*}\,Y_1^{\mu}\,Y_{lm}\,\mathrm d\Omega \ne 0
\quad\Longleftrightarrow\quad \Delta l=\pm1,\ \ \Delta m=\mu\in\{0,\pm1\} ,
```

with $\mu=0$ for $z$-polarized light and $\mu=\pm1$ for $x,y$-polarization. The
photon carries one unit of angular momentum, so $l$ must change by one and $m$ by
at most one. These are the **selection rules**, computed here by direct
integration.

- Reference: Sakurai & Napolitano (§5.6–5.7) and Griffiths {cite}`griffiths_qm`
  (time-dependent PT, the golden rule, atom–radiation interaction, selection
  rules); Nolting {cite}`nolting5b`. Cross-reference [§6.21](perturbation-fine-structure.ipynb) (time-independent PT),
  [§6.7](time-evolution.ipynb) (Rabi oscillations, the discrete two-level case), [§6.15](orbital-angular-momentum.ipynb) (spherical
  harmonics), [§6.17](hydrogen-atom.ipynb) (hydrogen orbitals), [§5.11](../05-classical-stat-mech/taste-of-nonequilibrium.ipynb) (irreversibility from reversible
  dynamics), and forward to [§6.25](bell-inequality.ipynb) (Movement VI) and Volume VII (the quantized
  field, spontaneous emission, Einstein coefficients — horizons).

---
## Setup

We work in units $\hbar=1$. The perturbation is switched on at $t=0$ and acts for
a time $T$; unless stated the matrix element $V_{0,fi}$ is a constant coupling
$g$. The reusable core:

- `first_order_amplitude(V_of_t, w_fi, T)`: the amplitude {eq}`eq-td-amplitude` as
  a time integral by `scipy.integrate.quad` (real and imaginary parts separately).
- `sinc2_lineshape(w, w_fi, T)`: the resonance factor $\mathrm{sinc}^2[(\omega-
  \omega_{fi})T/2]$ of {eq}`eq-td-resonance`.
- `continuum_probability(T, rho, band, g)`: the continuum sum
  $\int\rho(E)|c_f|^2\,\mathrm dE$ by `numpy.trapezoid` {eq}`eq-td-discrete-continuum`.
- `golden_rule_rate(V_fi, rho)`: $\Gamma=(2\pi/\hbar)|V_{fi}|^2\rho$
  {eq}`eq-golden-rule`.
- `dipole_angular_integral(lp, mp, l, m, mu)`: the three-$Y$ integral
  {eq}`eq-selection-rules` via `scipy.special.sph_harm_y`.

In [ ]:
import numpy as np
from scipy.integrate import quad
from scipy.special import sph_harm_y
import matplotlib.pyplot as plt

from ecp import draw, validate

HBAR = 1.0
SEL_TOL = 1e-3  # allowed/forbidden threshold for the dipole angular integrals


def first_order_amplitude(V_of_t, w_fi, T):
    r"""First-order transition amplitude {eq}`eq-td-amplitude` by ``scipy.integrate.quad``.

    Computes $c_f=-\tfrac{i}{\hbar}\int_0^T V(t')e^{i\omega_{fi}t'}\,\mathrm dt'$ as
    two real quadratures (the real and imaginary parts of the oscillatory integrand).

    Parameters
    ----------
    V_of_t : callable
        The matrix element $\langle f|V(t')|i\rangle$ as a function of $t'$.
    w_fi : float
        Transition frequency $(E_f-E_i)/\hbar$.
    T : float
        Duration the perturbation acts.

    Returns
    -------
    complex
        The amplitude $c_f(T)$.
    """
    re, _ = quad(lambda t: V_of_t(t) * np.cos(w_fi * t), 0.0, T, limit=200)
    im, _ = quad(lambda t: V_of_t(t) * np.sin(w_fi * t), 0.0, T, limit=200)
    return (-1j / HBAR) * (re + 1j * im)


def sinc2_lineshape(w, w_fi, T):
    r"""Resonance factor $\mathrm{sinc}^2[(\omega-\omega_{fi})T/2]$ {eq}`eq-td-resonance`.

    Uses ``numpy.sinc`` ($\mathrm{sinc}(x)=\sin(\pi x)/(\pi x)$), so the argument is
    divided by $\pi$.

    Parameters
    ----------
    w : float or numpy.ndarray
        Drive frequency (or array of frequencies).
    w_fi : float
        Transition frequency.
    T : float
        Duration.

    Returns
    -------
    float or numpy.ndarray
        The $\mathrm{sinc}^2$ lineshape, peaking at $w=w_{fi}$.
    """
    return np.sinc((w - w_fi) * T / 2.0 / np.pi) ** 2


def continuum_probability(T, rho, band, g=1.0):
    r"""Transition probability into a continuum by ``numpy.trapezoid`` {eq}`eq-td-discrete-continuum`.

    Sums the first-order $|c_f|^2=g^2T^2\,\mathrm{sinc}^2(\omega_{fi}T/2)$ over a band
    of final energies weighted by the density of states $\rho$.

    Parameters
    ----------
    T : float
        Duration.
    rho : float
        Density of final states (states per unit energy).
    band : numpy.ndarray
        Final-state energies relative to $E_i$ (so $\omega_{fi}=E$).
    g : float, optional
        The (constant) coupling matrix element.

    Returns
    -------
    float
        The total transition probability $P(T)$.
    """
    c2 = g**2 * T**2 * np.sinc(band * T / 2.0 / np.pi) ** 2
    return np.trapezoid(rho * c2, band)


def golden_rule_rate(V_fi, rho):
    r"""Fermi's golden rule rate $\Gamma=(2\pi/\hbar)|V_{fi}|^2\rho$ {eq}`eq-golden-rule`."""
    return 2.0 * np.pi / HBAR * np.abs(V_fi) ** 2 * rho


# A reusable spherical grid for the angular integrals (precomputed once).
_TH = np.linspace(0.0, np.pi, 240)
_PH = np.linspace(0.0, 2.0 * np.pi, 240)
_THg, _PHg = np.meshgrid(_TH, _PH, indexing="ij")
_SINT = np.sin(_THg)


def dipole_angular_integral(lp, mp, l, m, mu):
    r"""Dipole angular integral $\int Y_{l'm'}^{*}Y_1^{\mu}Y_{lm}\,\mathrm d\Omega$.

    The angular part of $\langle l'm'|\mathbf r|lm\rangle$ {eq}`eq-selection-rules`,
    evaluated on a spherical grid with ``scipy.special.sph_harm_y`` and integrated by
    ``numpy.trapezoid`` (with the $\sin\theta$ measure). Nonzero only for
    $\Delta l=\pm1$ and $\Delta m=\mu$.

    Parameters
    ----------
    lp, mp : int
        Final $(l',m')$.
    l, m : int
        Initial $(l,m)$.
    mu : int
        Photon component $\mu\in\{-1,0,1\}$ ($0$ for $z$, $\pm1$ for $x,y$).

    Returns
    -------
    float
        The magnitude of the angular integral.
    """
    integrand = (
        np.conj(sph_harm_y(lp, mp, _THg, _PHg))
        * sph_harm_y(1, mu, _THg, _PHg)
        * sph_harm_y(l, m, _THg, _PHg)
        * _SINT
    )
    inner = np.trapezoid(integrand, _PH, axis=1)  # over phi
    return float(np.abs(np.trapezoid(inner, _TH)))  # over theta

## Exercise 1 — The first-order transition amplitude

Derive and compute the first-order probability to transition between two states under a
perturbation switched on for a time $T$, and show the amplitude is the Fourier transform of the
matrix element at $\omega_{fi}$. Contrast a resonant transition (which accumulates) with an
off-resonant one (which does not).

1. Expand the state in $H_0$ eigenstates and write the first-order amplitude
   $c_f=-\tfrac{i}{\hbar}\int_0^T V_{fi}\,e^{i\omega_{fi}t'}\,\mathrm dt'$ from
   {eq}`eq-td-amplitude`.
2. For a constant coupling $V_{fi}=g$ switched on, evaluate the integral with
   `first_order_amplitude` (`scipy.integrate.quad`).
3. Compare $|c_f|^2$ to the closed form $g^2T^2\,\mathrm{sinc}^2(\omega_{fi}T/2)$.
4. Plot the running amplitude magnitude $|\int_0^t e^{i\omega_{fi}t'}\mathrm dt'|$:
   for $\omega_{fi}=0$ (resonant) it grows linearly; off resonance it merely
   oscillates and stays bounded.

Cite {eq}`eq-td-setup`, {eq}`eq-td-amplitude`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

The quadrature amplitude must reproduce the closed-form first-order probability
$g^2T^2\,\mathrm{sinc}^2(\omega_{fi}T/2)$: the transition amplitude is the Fourier
transform of $\langle f|V|i\rangle$ at $\omega_{fi}$.

In [ ]:
checks = []
for w_fi, T in [(0.5, 10.0), (1.3, 7.0), (2.0, 4.0)]:
    P_quad = abs(first_order_amplitude(lambda t: g, w_fi, T)) ** 2
    P_closed = g**2 * T**2 * np.sinc(w_fi * T / 2 / np.pi) ** 2
    checks.append(np.isclose(P_quad, P_closed, rtol=1e-4))
validate.check(
    all(checks),
    "the first-order transition probability from the quad time-integral equals "
    "g²T²·sinc²(ω_fi T/2) (the amplitude is the Fourier transform of ⟨f|V|i⟩)",
)

## Exercise 2 — Resonance and the sinc² lineshape

For a harmonic perturbation $V_0\cos(\omega t)$, show the transition probability peaks at
resonance $\omega=\omega_{fi}$ with a $\mathrm{sinc}^2$ lineshape whose width scales as
$\sim2\pi/T$.

1. Near resonance the probability is $P(\omega)\propto\mathrm{sinc}^2[(\omega-
   \omega_{fi})T/2]$ ({eq}`eq-td-resonance`); evaluate it with `sinc2_lineshape`
   over a range of drive frequencies.
2. Confirm the peak sits at $\omega=\omega_{fi}$ (energy conservation $\hbar\omega
   =E_f-E_i$).
3. Measure the full width at half maximum for several durations $T$.
4. Confirm the width scales as $1/T$ (the product $T\cdot\text{FWHM}$ is constant):
   the longer the perturbation acts, the sharper energy conservation becomes.

Cite {eq}`eq-td-resonance`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

The lineshape must peak at $\omega_{fi}$ and its width must scale as $1/T$, i.e.
the product $T\cdot\text{FWHM}$ is (nearly) constant across durations.

In [ ]:
tw = np.array([T * fw for T, fw in widths.items()])
validate.check(
    abs(peak_at - w_fi) < 0.02 and np.ptp(tw) / np.mean(tw) < 0.02,
    "P(ω) peaks at ω_fi with a sinc² lineshape whose FWHM scales as 1/T "
    "(T·FWHM constant): resonant transitions conserve energy",
)

## Exercise 3 — A discrete final state: the short-time Rabi connection

Show that first-order perturbation theory reproduces the short-time Rabi oscillation of [§6.7](time-evolution.ipynb), and
identify where it breaks down.

1. For a single resonant final state, the first-order probability is $P_1(t)=
   (\Omega t/2)^2$ with the Rabi frequency $\Omega=|V_{0,fi}|/\hbar$.
2. Compare to the *exact* two-level Rabi result $P(t)=\sin^2(\Omega t/2)$ ([§6.7](time-evolution.ipynb)).
3. Confirm they agree at short time ($\Omega t\ll1$), where $\sin^2(\Omega t/2)
   \approx(\Omega t/2)^2$.
4. Note that first-order theory fails once $P$ grows large — beyond that the exact
   treatment of [§6.7](time-evolution.ipynb) takes over, and the "probability" $P_1$ would exceed 1.

Cite {eq}`eq-td-discrete-continuum`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

At short time first-order theory and the exact Rabi oscillation must coincide:
$(\Omega t/2)^2\approx\sin^2(\Omega t/2)$ for $\Omega t\ll1$.

In [ ]:
validate.close(
    P_first[short],
    P_rabi[short],
    "first-order PT matches the short-time Rabi oscillation sin²(Ωt/2) (§6.7)",
    rtol=1e-2,
    atol=1e-3,
)

## Exercise 4 — Fermi's golden rule: a rate into a continuum

Show that transitions into a *continuum* of final states give a probability linear in time — a
constant rate — and derive Fermi's golden rule.

1. Take a band of final states of density $\rho$ around resonance, each coupled by
   the same $g$; sum the $\mathrm{sinc}^2$ lineshape over the band with
   `continuum_probability` (`numpy.trapezoid`).
2. Show $P(T)$ grows *linearly* in $T$ (so $P/T$ is constant), unlike the discrete
   $T^2$-then-oscillating behavior.
3. Extract the rate $P/T$ and confirm it equals $\Gamma=(2\pi/\hbar)|g|^2\rho$
   from `golden_rule_rate`.
4. Contrast on one plot with a single discrete (slightly detuned) state, whose
   probability merely oscillates.

Cite {eq}`eq-td-discrete-continuum`, {eq}`eq-golden-rule`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

Into a continuum the probability must be linear in time — a constant rate — equal
to Fermi's golden rule $\Gamma=(2\pi/\hbar)|g|^2\rho$. We check $P/T$ is constant
across durations and matches the formula.

In [ ]:
validate.close(
    rate_numeric[-1],
    rate_golden,
    "the continuum transition probability is linear in time — a constant rate "
    "Γ=(2π/ℏ)|g|²ρ (Fermi's golden rule)",
    rtol=2e-2,
)

## Exercise 5 — Absorption and emission of light

Set up the dipole interaction of an atom with an oscillating electric field, and identify resonant
absorption and stimulated emission as the golden-rule transitions whose matrix element is the
dipole moment.

1. In a field $\mathbf E\cos(\omega t)$ the perturbation is $V=-e\,\mathbf E\cdot
   \mathbf r\,\cos(\omega t)$, so the matrix element is the dipole moment
   $\langle f|\mathbf r|i\rangle$ ({eq}`eq-radiation`).
2. Resonant **absorption** occurs at $\hbar\omega=E_f-E_i>0$; **stimulated
   emission** at $E_f<E_i$ — the two resonances of {eq}`eq-td-resonance`.
3. The golden rule gives the rate, $\propto|\langle f|\mathbf r|i\rangle|^2$; a
   vanishing dipole moment means no (first-order) line.
4. Note that *spontaneous* emission — decay with no applied field — needs the
   quantized field (Volume VII horizon; the Einstein $A/B$ relation).

Cite {eq}`eq-radiation`, {eq}`eq-golden-rule`.

In [ ]:
# (solution hidden on the public site)


### Validation 5

The absorption and stimulated-emission rates are set by the same dipole matrix
element through the golden rule; a nonzero dipole moment gives a finite rate. We
check the golden-rule rate scales as $|\langle f|\mathbf r|i\rangle|^2$.

In [ ]:
d_small, d_big = 0.2, 0.6
validate.check(
    np.isclose(
        golden_rule_rate(d_big, rho) / golden_rule_rate(d_small, rho),
        (d_big / d_small) ** 2,
    ),
    "the absorption/emission rate scales as |⟨f|r|i⟩|² via the golden rule "
    "(light absorption and emission are dipole transitions at resonance)",
)

## Exercise 6 — Selection rules from the hydrogen orbitals

Compute the dipole angular integrals between angular-momentum states and show that transitions are
allowed only for $\Delta l=\pm1$ and $\Delta m=0,\pm1$ — the selection rules — directly from the
spherical harmonics, with no group theory.

1. The dipole components are proportional to $Y_1^\mu$ ($\mu=0$ for $z$, $\mu=\pm1$
   for $x,y$); the angular matrix element is $\int Y_{l'm'}^{*}Y_1^\mu Y_{lm}\,
   \mathrm d\Omega$ ({eq}`eq-selection-rules`).
2. Evaluate it for many $(l',m')\leftarrow(l,m)$ with `dipole_angular_integral`
   (`scipy.special.sph_harm_y`, `numpy.trapezoid`).
3. Classify each as allowed (magnitude above the tolerance $10^{-3}$) or forbidden.
4. Read off the pattern: nonzero only for $\Delta l=\pm1$, and $\Delta m=\mu$.

Cite {eq}`eq-selection-rules`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

Every allowed dipole transition must satisfy $\Delta l=\pm1$ and $\Delta m=0,\pm1$,
and no others: the selection rules follow from the spherical-harmonic integrals.

In [ ]:
validate.check(
    pattern_ok and allowed_mask.sum() > 0,
    "the dipole angular integrals are nonzero only for Δl=±1 and Δm=0,±1 "
    "(the selection rules follow from the Y-integrals, not group theory)",
)

## Exercise 7 — A forbidden transition, and why lines are missing *(student)*

Identify a transition the dipole selection rules forbid, show its matrix element vanishes, and
explain what that means for the spectrum — the metastable $2s$ state of hydrogen.

1. Pick a transition violating $\Delta l=\pm1$: the $2s\to1s$ decay has $\Delta l=0$
   ($l=0\to l'=0$).
2. Compute its dipole angular integral for every polarization and show all vanish.
3. Conclude the direct electric-dipole transition is *forbidden*, so $2s$ cannot
   decay this way and is very long-lived (metastable).
4. Note that higher-order and multipole processes allow it only weakly — which is
   exactly why some expected spectral lines are faint or absent.

Cite {eq}`eq-selection-rules`, {eq}`eq-golden-rule`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

A $\Delta l=0$ transition ($2s\to1s$) must have a vanishing dipole matrix element
for every polarization — forbidden — while the $\Delta l=\pm1$ neighbor ($2p\to1s$)
does not.

In [ ]:
validate.check(
    max(forbidden) < SEL_TOL < max(allowed_2p1s),
    "a Δl=0 transition (2s→1s) has vanishing dipole matrix element (forbidden), "
    "producing the metastable 2s state and missing lines",
)

## Exercise 8 — (Synthesis) The rule behind the spectrum

A perturbation that changes in time sets probability flowing between states,
favouring — ever more sharply the longer it acts — the transitions that conserve
energy. When the destination is a single level the flow sloshes back and forth,
reversible, the short-time face of Rabi. When it is a continuum, the sloshing
becomes a steady drain, and its rate is Fermi's golden rule: the coupling squared
times the density of places to go. That one crossover — from oscillation to
irreversible decay as one level becomes many — is the same passage from reversible
microscopics to irreversible macroscopics we watched in Volume V ([§5.11](../05-classical-stat-mech/taste-of-nonequilibrium.ipynb)), here in
its cleanest quantum form.

Applied to an atom in a light field, the coupling is the dipole moment, and the
geometry of the spherical harmonics decides which transitions are allowed: $\Delta
l=\pm1$, one unit of angular momentum handed to the photon. This is the machinery
of spectroscopy, and with it **Movement V is complete**. We have met the three
faces of approximation — perturbative ([§6.21](perturbation-fine-structure.ipynb)), variational ([§6.22](variational-method.ipynb)), semiclassical
([§6.23](wkb-semiclassical.ipynb)) — and now the dynamics of transitions that turns all of them toward
experiment. The golden rule looks almost too simple for how much it carries: a
matrix element, a density of states, a factor of $2\pi$. Yet every glowing gas,
every laser line, every fluorescent lifetime is that formula, and the reason a
neon sign is orange is a set of dipole integrals we just did by hand.

The final movement returns to the foundations — to the correlations that have no
classical explanation at all. The next notebook ([§6.25](bell-inequality.ipynb)) turns the Bell-state puzzle
of Movement I into a quantitative inequality, and watches quantum mechanics violate
it.

## Notebook summary

- **First-order amplitude** {eq}`eq-td-amplitude`: the transition amplitude is the
  Fourier transform of $\langle f|V|i\rangle$ at $\omega_{fi}$ (verified: the `quad`
  time-integral equals $g^2T^2\,\mathrm{sinc}^2$).
- **Resonance** {eq}`eq-td-resonance`: the $\mathrm{sinc}^2$ lineshape peaks at
  $\omega_{fi}$ and narrows as $2\pi/T$ — energy conservation emerging in time.
- **Discrete vs continuum** {eq}`eq-td-discrete-continuum`: first-order PT is the
  short-time face of Rabi ([§6.7](time-evolution.ipynb)); into a continuum the probability goes linear in $T$.
- **Fermi's golden rule** {eq}`eq-golden-rule`: $\Gamma=(2\pi/\hbar)|V_{fi}|^2\rho$,
  the numeric continuum rate matching the formula (`numpy.trapezoid` over the band).
- **Selection rules** {eq}`eq-selection-rules`: the dipole angular integrals
  (`scipy.special.sph_harm_y`) are nonzero only for $\Delta l=\pm1$, $\Delta m=0,\pm1$;
  the $\Delta l=0$ $2s\to1s$ integral vanishes — the metastable $2s$.

## Outlook

- **Bell's inequality** ([§6.25](bell-inequality.ipynb)): the Movement I entanglement puzzle made
  quantitative, and its quantum violation computed — opening Movement VI.
- **The quantized radiation field**, spontaneous emission, and the Einstein $A/B$
  coefficients; natural line widths and lifetimes (Volume VII; horizons).
- **Scattering as a golden-rule process** and the Born approximation (a horizon).
- Cross-reference [§6.21](perturbation-fine-structure.ipynb) (time-independent PT), [§6.7](time-evolution.ipynb) (Rabi), [§6.15](orbital-angular-momentum.ipynb) (spherical
  harmonics), [§6.17](hydrogen-atom.ipynb) (hydrogen orbitals), [§5.11](../05-classical-stat-mech/taste-of-nonequilibrium.ipynb) (irreversibility), forward to [§6.25](bell-inequality.ipynb) and
  Volume VII.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()